First of all we need some dependencies:

In [109]:
#Dependencies, comment out install comments fpr use with GPU

# Import PyTorch and other relevant libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Sampler, random_split

#Numpy, Matplotlib and random
import numpy as np
import numpy.random as rd
import matplotlib.pyplot as plt
import random

#Nice Visualisation
from tqdm import tqdm

#Python OS module
import os

#Pandas to extract Data
import pandas as pd

#For preprocessing the date
from sklearn import preprocessing as pp
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

In [2]:
from google.colab import drive
drive.mount('/content/drive')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Mounted at /content/drive


In [ ]:
from re import L
#using our label conversion if needed

def label_conversion(label_raw):
  label = (label_raw != 0).to(dtype=torch.float32)
  return label


def convert_label_to_binary(label):
    if pd.isna(label):
        return [0, 0, 0, 0]

    binary_mapping = {
        0: [0, 0, 0, 0],
        1: [0, 0, 1, 0],
        2: [0, 0, 0, 1],
        3: [0, 0, 1, 1],
        4: [1, 0, 0, 0],
        5: [0, 1, 0, 0],
        6: [1, 1, 0, 0],
        7: [1, 0, 1, 0],
        8: [0, 1, 0, 1],
        9: [1, 0, 0, 1],
        10: [0, 1, 1, 0],
        11: [1, 1, 1, 1]
    }
    return binary_mapping[label]

def binary_to_original_label(binary_labels):
    label_mapping = {
        (0, 0, 0, 0): 0,
        (0, 0, 1, 0): 1,
        (0, 0, 0, 1): 2,
        (0, 0, 1, 1): 3,
        (1, 0, 0, 0): 4,
        (0, 1, 0, 0): 5,
        (1, 1, 0, 0): 6,
        (1, 0, 1, 0): 7,
        (0, 1, 0, 1): 8,
        (1, 0, 0, 1): 9,
        (0, 1, 1, 0): 10,
        (1, 1, 1, 1): 11
    }

    original_labels = [label_mapping.get(tuple(map(int, row)), -1) for row in binary_labels]

    return torch.tensor(original_labels)

#Data to train and test
df = pd.read_pickle('../../data/train.pickle')

#import Data
sensor_data = torch.tensor(df['sensor_data'],dtype=torch.float32)
label_raw = torch.tensor(df['label'])
label_conv = label_conversion(label_raw)

#test if the modell predicts something
test = sensor_data[:10]


Explanation:

Please import your best modell (.pth file) to my [drive](https://drive.google.com/drive/folders/182e-XeLpLMKeflvn6FlqoSyqgMHIMYPM?usp=sharing) naming it (2DCNN, RNN,...)

Copy the block and fill it out:



In [ ]:
# Load your modell

# -- insert modell class here --

# FILL: .......

# load your modell by instanciate a class (we need to know the parameter)

# FILL: modell_NAME =  modellclass(input_dim, hidden_size, batch_size, num_layers, dropout).to(device) #change to necessary parameter
# FILL: modell_NAME.load_state_dict(torch.load('/content/drive/MyDrive/training_checkpoints/modell_NAME', weights_only=False))

#if you modiefied the measurement data (standardizing/normalization) please specify

# FILL: def LSTM_prediction(input):
#     function that return prediction for measurement

# FILL: convert label if necessary







Example:

In [ ]:
# Load your modell

# -- insert modell class here --

class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_size, batch_size, num_layers, dropout):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_dim, hidden_size, num_layers=num_layers, dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def init_hidden(self, batch_size, device):
        return (torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device),
                torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device))

    def forward(self, x, state=None, return_state=False):
        if state is None:
            state = self.init_hidden(x.size(0), x.device)
        out, state = self.lstm(x, state)
        out = self.fc(out[:, -1, :])
        return out if not return_state else (out, state)

# load your modell by instanciate a class (we need to know the parameter)

# ---- LSTM with learning rate 10^-2 ----


LSTM1 = LSTMModel(input_dim=6, hidden_size=512, batch_size=64, num_layers=1, dropout=0).to(device)
LSTM1.load_state_dict(torch.load('../LSTM/Models/LSTM_lr-2', map_location=torch.device('cpu') ))
LSTM1.eval()

def LSTM1_prediction(input):
  # make prediction
  test_input = input #test.reshape(input.size(0), 128, 6).to(device)

  # Vorhersage
  with torch.no_grad():
      y = LSTM1(test_input)

  probs = torch.sigmoid(y)
  pred = (probs > 0.5).long()

  return probs # pred.squeeze().item()

# ---- LSTM trained with hard coded data augmentation

LSTM2 = LSTMModel(input_dim=6, hidden_size=512, batch_size=64, num_layers=1, dropout=0).to(device)
LSTM2.load_state_dict(torch.load('../LSTM/Models/LSTM_hardcodedaug', map_location=torch.device('cpu') ))
LSTM2.eval()

def LSTM2_prediction(input):
  # make prediction
  test_input = input #test.reshape(input.size(0), 128, 6).to(device)

  # Vorhersage
  with torch.no_grad():
      y = LSTM2(test_input)

  probs = torch.sigmoid(y)
  pred = (probs > 0.5).long()

  return probs # pred.squeeze().item()


# ---- LSTM with data generated from GAN ----

LSTM3 = LSTMModel(input_dim=6, hidden_size=512, batch_size=64, num_layers=1, dropout=0).to(device)
LSTM3.load_state_dict(torch.load('../LSTM/Models/LSTM_GAN', map_location=torch.device('cpu') ))
LSTM3.eval()

def LSTM3_prediction(input):
  # make prediction
  test_input = input #test.reshape(input.size(0), 128, 6).to(device)

  # Vorhersage
  with torch.no_grad():
      y = LSTM3(test_input)

  probs = torch.sigmoid(y)
  pred = (probs > 0.5).long()

  return probs # pred.squeeze().item()


# ---- LSTM with data generated from TGAN ----

LSTM4 = LSTMModel(input_dim=6, hidden_size=512, batch_size=64, num_layers=1, dropout=0).to(device)
LSTM4.load_state_dict(torch.load('../LSTM/Models/LSTM_TGAN', map_location=torch.device('cpu') ))
LSTM4.eval()

def LSTM4_prediction(input):
  # make prediction
  test_input = input #test.reshape(input.size(0), 128, 6).to(device)

  # Vorhersage
  with torch.no_grad():
      y = LSTM4(test_input)

  probs = torch.sigmoid(y)
  pred = (probs > 0.5).long()

  return probs # pred.squeeze().item()


# print(LSTM1_prediction(test, 1))
# print(LSTM2_prediction(test, 1))
# print(LSTM3_prediction(test, 1))
# print(LSTM4_prediction(test, 1))


We generate an input for a higher order neural network based on the prediciton on a measurement

In [89]:
def cat_prediction(batch):
 pred = torch.stack([
      LSTM1_prediction(batch),
      LSTM2_prediction(batch),
      LSTM3_prediction(batch),
      LSTM4_prediction(batch)
  ], dim=1)
 return pred

prediction = cat_prediction(sensor_data)

print(prediction.shape)


torch.Size([4053, 4, 1])


Split the Data in Train/Val Data

In [90]:
pred = prediction.squeeze(-1)

# Dataset erstellen
dataset = TensorDataset(pred, label_conv)

# Dataloader erstellen
batch_size = 32
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


FCN Modell

In [91]:
class FCN(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=32, output_dim=2):
        super(FCN, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_dim, output_dim)  # output_dim = 2 für binär

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)  # Kein Softmax – CrossEntropyLoss erwartet raw logits
        return x

Initialize

In [95]:
# Beispielhafte Initialisierung
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FCN().to(device)

total = label_conv.size(0)

weight0 = np.bincount(label_conv)[0].item()/total
weight1 = np.bincount(label_conv)[1].item()/total

weights = torch.tensor([weight0, weight1], dtype=torch.float32).to(device)

print(weights)

criterion = nn.CrossEntropyLoss(weight=weights)  # Erwartet target: [batch], mit Klassen-IDs 0 oder 1
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

tensor([0.4436, 0.5564])


Now we train our easy modell

In [ ]:
for epoch in range(100):  # Anzahl Epochen
    model.train()
    total_loss = 0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)           # Shape: [b, 4]
        y_batch = y_batch.long().to(device)    # Shape: [b], dtype long für CE-Loss

        optimizer.zero_grad()
        outputs = model(x_batch)               # Shape: [b, 2]
        loss = criterion(outputs, y_batch)     # Ziel: Index 0 oder 1
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Now we make a prediction

In [ ]:
def predict(model, x):
    model.eval()
    with torch.no_grad():
        logits = model(x.to(device))
        preds = torch.argmax(logits, dim=1)
    return preds


df_test = pd.read_pickle('../../data/test.pickle')

#import Data
test_sensor_data = torch.tensor(df['sensor_data'],dtype=torch.float32)
test_label = torch.tensor(df['label'])
test_label_conv = label_conversion(test_label)

print(test_sensor_data.shape)

# Dataset erstellen
test_data = TensorDataset(test_sensor_data, test_label_conv)

# Dataloader erstellen
batch_size = 32
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=True)

correct = 0
total = 0

all_preds = []
all_labels = []

for x, y in tqdm(test_loader, desc="Evaluating"):
    pred_raw = cat_prediction(x)
    pred_raw = pred_raw.squeeze(-1)
    pred = predict(model, pred_raw)

    correct += (pred == y).sum().item()
    total += y.size(0)

    all_preds.append(pred.cpu())
    all_labels.append(y.cpu())

accuracy = correct / total
print(f"Accuracy: {accuracy:.4f}")

# Alle Predictions und Labels concatenaten
all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

# Confusion Matrix berechnen
cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(cm)


torch.Size([4053, 128, 6])


Evaluating: 100%|██████████| 127/127 [03:04<00:00,  1.45s/it]

Accuracy: 0.8263
Confusion Matrix:
[[1531  267]
 [ 437 1818]]
